<!-- DOC-BANNER -->
# ColumnTransformer (Day 28)

> **Playlist:** CampusX Day 28 *Column Transformer in Machine Learning / How to use ColumnTransformer in Sklearn*.
> **Purpose:** Same preprocessing two ways on `covid_toy.csv` (`age/gender/fever/cough/city/has_covid`): WITHOUT-CT manual (`SimpleImputer(fever)` → `OrdinalEncoder(['Mild','Strong'])` on cough → `OneHotEncoder(drop='first')` on gender/city → passthrough age → `np.concatenate` (80,7)) vs WITH-CT (`ColumnTransformer(tnf1: impute fever, tnf2: ordinal cough, tnf3: one-hot age+gender, remainder='passthrough')` → `fit_transform` train/test (20,23)).
> **After studying:** you can bundle per-column steps into one transformer.


## `ColumnTransformer`: all column steps in one object

Models need numeric, non-missing input. That means one-hot/ordinal encoding for categories and imputation for NaNs — the column transformer runs every per-column step in one go.


In [ ]:
import matplotlib.pyplot as plt 
import  seaborn as sns 
import pandas as pd 
import numpy as  np 

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(
    style="darkgrid",
    context="notebook",
    palette="husl",
    font="sans-serif",
    font_scale=1.1,
    rc={
        "figure.figsize": (10, 6),
        "figure.dpi": 120,
        "axes.titlesize": 18,
        "axes.titleweight": "bold",
        "axes.labelsize": 12,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.alpha": 0.25,
        "lines.linewidth": 2.5,
        "lines.markersize": 7,
        "legend.frameon": False,
        "figure.autolayout": True
    }
)

plt.rcParams["figure.facecolor"] = "#1e1e2f"
plt.rcParams["axes.facecolor"] = "#282840"
plt.rcParams["axes.labelcolor"] = "#eeeeff"
plt.rcParams["text.color"] = "#eeeeff"
plt.rcParams["xtick.color"] = "#ccccdd"
plt.rcParams["ytick.color"] = "#ccccdd"
plt.rcParams["grid.color"] = "#777799"

In [ ]:
df = pd.read_csv(r"E:\MACHINE_LEARNING\DATA_SETS\covid_toy.csv")
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
from sklearn.model_selection import train_test_split 

X_train ,X_test , y_train , y_test = train_test_split(df.drop(columns=  ['has_covid']) , df['has_covid'] , test_size = 0.2)

### Manual path: without `ColumnTransformer`

Each block (impute, ordinal, one-hot) is applied by hand and glued with `np.concatenate` — you track the column order yourself.


In [ ]:
from sklearn.impute import SimpleImputer 


In [ ]:

si = SimpleImputer()

X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever

In [ ]:
# ORDINAL ENCODING -> cough 

from sklearn.preprocessing  import OrdinalEncoder as OE


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=[["Mild", "Strong"]])

X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.transform(X_test[['cough']])

### One-hot encoding gender and city

Unordered text columns become binary columns (`drop="first"` keeps n-1 and avoids the dummy trap).


In [ ]:
from sklearn.preprocessing import OneHotEncoder


In [ ]:
ohe = OneHotEncoder(drop = 'first' ,sparse_output= False ,dtype= np.int32)

In [ ]:
X_train_gender_city = ohe.fit_transform(X_train[['gender' , 'city']])


# also the test data 


X_test_gender_city = ohe.transform(X_test[['gender' , 'city']])


In [ ]:
X_train_gender_city 

In [ ]:
X_train_age= X_train.drop(columns= ['gender' , 'fever' ,  'cough' , 'city']).values


X_test_age= X_test.drop(columns= ['gender' , 'fever' ,  'cough' , 'city']).values


In [71]:
X_train_transformed = np.concatenate((X_train_age , X_train_gender_city , X_train_cough ,X_train_fever ), axis = 1 )

X_test_transformed = np.concatenate((X_test_age , X_test_gender_city , X_test_cough ,X_test_fever ), axis = 1 )

X_train_transformed.shape


(80, 7)

### Automated path: with `ColumnTransformer`

The same steps declared as (name, transformer, columns) triples — one `fit_transform` returns the finished array.


### WITHOUT vs WITH `ColumnTransformer`
Manual path concatenates transformed blocks by hand (error-prone column tracking); `ColumnTransformer` declares (name, transformer, columns) triples and returns one array. Note the column mismatch preserved: manual uses gender+city for OneHot, CT uses age+gender — compare carefully. Only tail rows executed (71,78,79,81); rest `null`.

In [78]:
from sklearn.compose import ColumnTransformer

In [79]:
transformer = ColumnTransformer(transformers =[
    
    ('tnf1', SimpleImputer() , ['fever']) ,
    ('tnf2' , OrdinalEncoder(categories= [['Mild' , 'Strong']]),['cough']),
    ('tnf3 ' ,OneHotEncoder(sparse_output = False , drop= 'first') ,['age' ,'gender'])] , remainder= 'passthrough')

In [81]:
transformer.fit_transform(X_train).shape

transformer.fit_transform(X_test).shape

(20, 23)

<!-- EXPERIMENT -->
### Experiment: fix the leak, compare the arrays
Replace the test-side `fit_transform` with `transform` and predict whether the output shape changes (it should not) and whether the *values* can change (they can — different learned parameters). Same shape, different numbers: that is data leakage made visible.

<!-- DOC-BANNER -->
## Key takeaways (Day 28)
- `ColumnTransformer` = per-column pipelines + single `fit_transform`/`transform`.
- **Bugs preserved:** `fit_transform` on `X_test` twice (must be `transform` — else leakage + different categories); no model/eval; `train_test_split(test .2, no random_state)` is non-reproducible; double `plt/sns` import; dark-theme boilerplate shared with 4 notebooks.
- **ML connection:** this is the `trf1` inside [Day-29 pipelines](../Day_29/topic_notes.md) — learn it here, automate it there.
